# Task 122: impedance_eis — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Electrochemical impedance spectroscopy fitting using Randles circuit

**Paper**: impedance.py: A Python package for electrochemical impedance analysis
**Repository**: https://github.com/ECSHackWeek/impedance.py

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 57.01 dB
- **SSIM**: N/A (1D spectral fitting)
- **Evaluation**: EIS equivalent circuit parameter fitting via nonlinear least squares (PSNR + CC + parameter RE)

### Mathematical Formulation

**Blind Source Separation**: observed mixtures $\mathbf{x}(t) = A\mathbf{s}(t)$

**ICA** finds unmixing $W$: $\hat{\mathbf{s}} = W\mathbf{x}$ maximizing non-Gaussianity:
$$\max_{\mathbf{w}} |E\{G(\mathbf{w}^T\mathbf{x})\} - E\{G(\nu)\}|$$

**NMF**: $V \approx WH$, with $W, H \geq 0$, minimizing $D(V\|WH)$.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 122: Electrochemical Impedance Spectroscopy (EIS) Fitting
Inverse Problem: Fit equivalent circuit model (Randles circuit) to measured
complex impedance spectra Z(ω) to recover circuit parameters.

Circuit: R0-p(R1,C1)-W  (Randles circuit)
  R0 = series/ohmic resistance
  R1 = charge transfer resistance
  C1 = double-layer capacitance
  σ_W = Warburg coefficient

Z(ω) = R0 + Z_RC(ω) + Z_W(ω)
where Z_RC = R1 / (1 + jωR1C1)   (parallel RC)
      Z_W  = σ_W / sqrt(ω) * (1 - j)  (Warburg impedance)
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`randles_impedance`, `objective`

In [ ]:
# ──────────────────────────────────────────────────────────
# 2. Forward model
# ──────────────────────────────────────────────────────────
def randles_impedance(freq, R0, R1, C1, sigma_W):
    """Compute complex impedance of a Randles circuit with Warburg element.

    Parameters
    ----------
    freq : ndarray
        Frequency array in Hz.
    R0, R1, C1, sigma_W : float
        Circuit parameters.

    Returns
    -------
    Z : ndarray (complex)
        Complex impedance at each frequency.
    """
    omega = 2.0 * np.pi * freq
    # Parallel RC element: Z_RC = R1 / (1 + j*omega*R1*C1)
    Z_RC = R1 / (1.0 + 1j * omega * R1 * C1)
    # Warburg element: Z_W = sigma_W / sqrt(omega) * (1 - j)
    Z_W = sigma_W / np.sqrt(omega) * (1.0 - 1j)
    # Total impedance
    Z = R0 + Z_RC + Z_W
    return Z

# ──────────────────────────────────────────────────────────
# 5. Inverse solver – nonlinear least squares via L-BFGS-B
# ──────────────────────────────────────────────────────────
def objective(params_vec, freq, Z_meas):
    """Weighted least-squares objective on real + imaginary parts."""
    Z_model = forward(params_vec, freq)
    # Weight by 1/|Z_meas| so all frequencies contribute equally
    w = 1.0 / np.abs(Z_meas)
    residual_re = (Z_model.real - Z_meas.real) * w
    residual_im = (Z_model.imag - Z_meas.imag) * w
    return 0.5 * np.sum(residual_re**2 + residual_im**2)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `forward`

In [ ]:
def forward(params_vec, freq):
    """Wrapper: parameter vector -> complex impedance."""
    R0, R1, C1, sigma_W = params_vec
    return randles_impedance(freq, R0, R1, C1, sigma_W)

## 5. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import json
import os

# ──────────────────────────────────────────────────────────

### 1. Ground truth parameters

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Ground truth parameters
# ──────────────────────────────────────────────────────────
GT_PARAMS = {
    'R0': 50.0,       # Ohm  – series resistance
    'R1': 100.0,      # Ohm  – charge transfer resistance
    'C1': 1e-6,       # F    – double-layer capacitance
    'sigma_W': 50.0,  # Ohm·s^(-1/2) – Warburg coefficient
}

PARAM_ORDER = ['R0', 'R1', 'C1', 'sigma_W']

# ──────────────────────────────────────────────────────────

### 2. Forward model

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward model
# ──────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────

### 3. Frequency grid & ground truth spectrum

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 3. Frequency grid & ground truth spectrum
# ──────────────────────────────────────────────────────────
freq = np.logspace(-2, 6, 60)  # 0.01 Hz to 1 MHz, 60 points

gt_vec = np.array([GT_PARAMS[k] for k in PARAM_ORDER])
Z_true = forward(gt_vec, freq)

# ──────────────────────────────────────────────────────────

### 4. Add noise (1.5% of |Z| Gaussian noise to Re and Im)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 4. Add noise (1.5% of |Z| Gaussian noise to Re and Im)
# ──────────────────────────────────────────────────────────
np.random.seed(42)
noise_level = 0.015
noise_re = noise_level * np.abs(Z_true) * np.random.randn(len(freq))
noise_im = noise_level * np.abs(Z_true) * np.random.randn(len(freq))
Z_noisy = Z_true + noise_re + 1j * noise_im

# ──────────────────────────────────────────────────────────

### 5. Inverse solver – nonlinear least squares via L-BFGS-B

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 5. Inverse solver – nonlinear least squares via L-BFGS-B
# ──────────────────────────────────────────────────────────


# Initial guesses – try multiple starting points to avoid local minima
bounds = [(1.0, 500.0),    # R0
          (1.0, 1000.0),   # R1
          (1e-9, 1e-3),    # C1
          (1.0, 500.0)]    # sigma_W

initial_guesses = [
    np.array([30.0, 150.0, 5e-6, 80.0]),
    np.array([60.0, 80.0, 2e-6, 30.0]),
    np.array([40.0, 120.0, 8e-7, 60.0]),
    np.array([70.0, 50.0, 3e-6, 40.0]),
    np.array([20.0, 200.0, 1e-5, 20.0]),
]

best_result = None
best_cost = np.inf

for x0 in initial_guesses:
    res = minimize(objective, x0, args=(freq, Z_noisy),
                   method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 10000, 'ftol': 1e-18, 'gtol': 1e-14})
    if res.fun < best_cost:
        best_cost = res.fun
        best_result = res

# Refine with Nelder-Mead (derivative-free, good for noisy landscapes)
from scipy.optimize import differential_evolution

de_result = differential_evolution(objective, bounds, args=(freq, Z_noisy),
                                   seed=42, maxiter=2000, tol=1e-14,
                                   polish=True, workers=1)
if de_result.fun < best_cost:
    best_cost = de_result.fun
    best_result = de_result

result = best_result
fitted_params = result.x
Z_fitted = forward(fitted_params, freq)

print("Optimization converged:", result.success)
print(f"Objective value: {result.fun:.6e}")

# ──────────────────────────────────────────────────────────

### 6. Metrics

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 6. Metrics
# ──────────────────────────────────────────────────────────
# Parameter relative errors
param_errors = {}
for i, name in enumerate(PARAM_ORDER):
    gt_val = gt_vec[i]
    fit_val = fitted_params[i]
    rel_err = np.abs(fit_val - gt_val) / np.abs(gt_val)
    param_errors[name] = {
        'ground_truth': float(gt_val),
        'fitted': float(fit_val),
        'relative_error': float(rel_err),
    }
    print(f"  {name}: GT={gt_val:.4e}, Fit={fit_val:.4e}, RelErr={rel_err:.4f}")

# Spectral RMSE (on complex impedance)
rmse = np.sqrt(np.mean(np.abs(Z_fitted - Z_true)**2))

# Spectral PSNR
max_Z = np.max(np.abs(Z_true))
psnr = 20.0 * np.log10(max_Z / rmse) if rmse > 0 else float('inf')

# Correlation coefficient between fitted and true spectra
# Treat as 1D real signal: concatenate real and imag parts
sig_true = np.concatenate([Z_true.real, Z_true.imag])
sig_fit = np.concatenate([Z_fitted.real, Z_fitted.imag])
cc = float(np.corrcoef(sig_true, sig_fit)[0, 1])

# Mean parameter relative error
mean_re = float(np.mean([v['relative_error'] for v in param_errors.values()]))

print(f"\nSpectral RMSE: {rmse:.4f} Ohm")
print(f"Spectral PSNR: {psnr:.2f} dB")
print(f"Correlation Coefficient: {cc:.6f}")
print(f"Mean Parameter Relative Error: {mean_re:.4f}")

# ──────────────────────────────────────────────────────────

### 7. Save results

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 7. Save results
# ──────────────────────────────────────────────────────────
os.makedirs('results', exist_ok=True)

metrics = {
    'task': 'impedance_eis',
    'task_number': 122,
    'description': 'Electrochemical Impedance Spectroscopy (EIS) Randles circuit fitting',
    'psnr_db': round(float(psnr), 2),
    'rmse_ohm': round(float(rmse), 4),
    'correlation_coefficient': round(cc, 6),
    'mean_parameter_relative_error': round(mean_re, 6),
    'parameters': param_errors,
    'noise_level_percent': noise_level * 100,
    'num_frequencies': len(freq),
    'optimizer': 'L-BFGS-B',
    'converged': bool(result.success),
}

with open('results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

# Save spectra as npy (stacked real+imag, shape 2×N)
np.save('results/ground_truth.npy', np.stack([Z_true.real, Z_true.imag]))
np.save('results/reconstruction.npy', np.stack([Z_fitted.real, Z_fitted.imag]))

# ──────────────────────────────────────────────────────────

### 8. Visualization

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 8. Visualization
# ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Task 122: EIS Randles Circuit Fitting', fontsize=15, fontweight='bold')

### (a) Nyquist plot ---

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# --- (a) Nyquist plot ---
ax = axes[0, 0]
ax.plot(Z_true.real, -Z_true.imag, 'b-', linewidth=2, label='Ground Truth')
ax.plot(Z_noisy.real, -Z_noisy.imag, 'k.', markersize=5, alpha=0.5, label='Noisy Data')
ax.plot(Z_fitted.real, -Z_fitted.imag, 'r--', linewidth=2, label='Fitted')
ax.set_xlabel('Z_real (Ω)', fontsize=11)
ax.set_ylabel('-Z_imag (Ω)', fontsize=11)
ax.set_title('(a) Nyquist Plot', fontsize=12)
ax.legend(fontsize=9)
ax.set_aspect('equal', adjustable='datalim')
ax.grid(True, alpha=0.3)

### (b) Bode plot: |Z| and phase vs frequency ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- (b) Bode plot: |Z| and phase vs frequency ---
ax = axes[0, 1]
ax.loglog(freq, np.abs(Z_true), 'b-', linewidth=2, label='|Z| GT')
ax.loglog(freq, np.abs(Z_noisy), 'k.', markersize=4, alpha=0.4, label='|Z| Noisy')
ax.loglog(freq, np.abs(Z_fitted), 'r--', linewidth=2, label='|Z| Fitted')
ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('|Z| (Ω)', fontsize=11)
ax.set_title('(b) Bode Plot – Magnitude', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')

# Phase on twin axis
ax2 = ax.twinx()
phase_true = np.degrees(np.angle(Z_true))
phase_fitted = np.degrees(np.angle(Z_fitted))
ax2.semilogx(freq, phase_true, 'b:', linewidth=1.5, alpha=0.6)
ax2.semilogx(freq, phase_fitted, 'r:', linewidth=1.5, alpha=0.6)
ax2.set_ylabel('Phase (°)', fontsize=10, color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

### (c) Parameter comparison bar chart ---

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# --- (c) Parameter comparison bar chart ---
ax = axes[1, 0]
names = PARAM_ORDER
gt_vals_norm = []
fit_vals_norm = []
for i, name in enumerate(names):
    gt_v = gt_vec[i]
    fit_v = fitted_params[i]
    # Normalise to GT for visual comparison
    gt_vals_norm.append(1.0)
    fit_vals_norm.append(fit_v / gt_v)

x_pos = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x_pos - width/2, gt_vals_norm, width, label='Ground Truth', color='steelblue')
bars2 = ax.bar(x_pos + width/2, fit_vals_norm, width, label='Fitted', color='salmon')
ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('Normalised Value (GT = 1)', fontsize=11)
ax.set_title('(c) Parameter Recovery (normalised)', fontsize=12)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

# Add relative error annotation
for i, name in enumerate(names):
    re = param_errors[name]['relative_error']
    ax.text(x_pos[i] + width/2, fit_vals_norm[i] + 0.02,
            f'{re:.1%}', ha='center', va='bottom', fontsize=8, color='red')

### (d) Residuals plot ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- (d) Residuals plot ---
ax = axes[1, 1]
residual_re = Z_fitted.real - Z_true.real
residual_im = Z_fitted.imag - Z_true.imag
ax.semilogx(freq, residual_re, 'b.-', markersize=4, label='ΔZ_real', alpha=0.7)
ax.semilogx(freq, residual_im, 'r.-', markersize=4, label='ΔZ_imag', alpha=0.7)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('Residual (Ω)', fontsize=11)
ax.set_title('(d) Fit Residuals (Fitted − GT)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('results/reconstruction_result.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"\n✓ All results saved to results/")
print(f"  metrics.json, reconstruction_result.png, ground_truth.npy, reconstruction.npy")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 6. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **impedance_eis**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=57.01 dB, SSIM=N/A (1D spectral fitting))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: impedance.py: A Python package for electrochemical impedance analysis
- Repository: https://github.com/ECSHackWeek/impedance.py
